# Big Six Valuation Plots

This notebook visualises 2025/26 model valuations against current Transfermarkt values for Big Six Premier League players.

The dashed diagonal is the fair-value line. Players above it have model values higher than Transfermarkt values; players below it have model values lower than Transfermarkt values.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## Load scoring results

In [ ]:
scoring_results = pd.read_csv("../data/processed/scoring_results_2025_26.csv")

figures_dir = Path("../reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

scoring_results.shape, scoring_results.columns.tolist()

## Filter Big Six players

Use `season_club_name` rather than `current_club_name`, because the plot should reflect the club a player appeared for during the 2025/26 Premier League season.

In [ ]:
big_six_clubs = [
    "Arsenal FC",
    "Chelsea FC",
    "Liverpool FC",
    "Manchester City",
    "Manchester United",
    "Tottenham Hotspur",
]

big_six = scoring_results.loc[
    scoring_results["season_club_name"].isin(big_six_clubs)
    & scoring_results["current_market_value_in_eur"].notna()
    & scoring_results["predicted_value"].notna()
].copy()

big_six["current_value_m"] = big_six["current_market_value_in_eur"] / 1_000_000
big_six["predicted_value_m"] = big_six["predicted_value"] / 1_000_000
big_six["valuation_gap_m"] = big_six["valuation_gap"] / 1_000_000

big_six["season_club_name"].value_counts().reindex(big_six_clubs)

## Most model-undervalued Big Six players

In [ ]:
most_undervalued_by_club = (
    big_six.loc[big_six["minutes_band"].eq("regular_minutes")]
    .sort_values("valuation_gap", ascending=False)
    .groupby("season_club_name", as_index=False)
    .head(1)
    [
        [
            "season_club_name",
            "player_name",
            "position",
            "sub_position",
            "minutes_played",
            "current_value_m",
            "predicted_value_m",
            "valuation_gap_m",
            "has_previous_market_value",
        ]
    ]
    .sort_values("season_club_name")
)

most_undervalued_by_club

## Plot helpers

In [ ]:
def format_eur_m(value):
    sign = "+" if value > 0 else ""
    return f"{sign}EUR {value:.0f}m"


def club_gap_extremes(data, club_name, *, top_each=3):
    club_data = data.loc[data["season_club_name"].eq(club_name)].copy()

    positive = (
        club_data.loc[club_data["valuation_gap_m"].gt(0)]
        .sort_values("valuation_gap_m", ascending=False)
        .head(top_each)
    )
    negative = (
        club_data.loc[club_data["valuation_gap_m"].lt(0)]
        .sort_values("valuation_gap_m", ascending=True)
        .head(top_each)
    )
    return positive, negative


def format_summary_player(row):
    return f"{row['player_name']} ({format_eur_m(row['valuation_gap_m'])})"


def plot_club_valuation_scatter(
    data,
    club_name,
    *,
    top_each=2,
    min_minutes_for_labels=450,
):
    club_data = data.loc[data["season_club_name"].eq(club_name)].copy()

    max_value = max(
        club_data["current_value_m"].max(),
        club_data["predicted_value_m"].max(),
    )
    axis_limit = max_value * 1.12

    fig, ax = plt.subplots(figsize=(8, 8.7))

    ax.fill_between(
        [0, axis_limit],
        [0, axis_limit],
        [axis_limit, axis_limit],
        color="#dff3e6",
        alpha=0.45,
    )
    ax.fill_between(
        [0, axis_limit],
        [0, 0],
        [0, axis_limit],
        color="#f7dddd",
        alpha=0.45,
    )
    ax.plot(
        [0, axis_limit],
        [0, axis_limit],
        color="#222222",
        linestyle="--",
        linewidth=1.2,
    )
    ax.text(
        axis_limit * 0.62,
        axis_limit * 0.68,
        "fair-value line",
        rotation=36,
        fontsize=9,
        color="#333333",
    )

    undervalued = club_data.loc[club_data["valuation_gap_m"].ge(0)]
    overvalued = club_data.loc[club_data["valuation_gap_m"].lt(0)]

    ax.scatter(
        undervalued["current_value_m"],
        undervalued["predicted_value_m"],
        s=undervalued["minutes_played"].clip(lower=90) / 14,
        c="#128a43",
        marker="^",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.9,
    )
    ax.scatter(
        overvalued["current_value_m"],
        overvalued["predicted_value_m"],
        s=overvalued["minutes_played"].clip(lower=90) / 14,
        c="#c73636",
        marker="v",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.9,
    )

    positive_labels, negative_labels = club_gap_extremes(
        club_data,
        club_name,
        top_each=top_each,
    )

    for _, row in positive_labels.iterrows():
        label = f"{row['player_name']}\n{format_eur_m(row['valuation_gap_m'])}"
        if row["minutes_played"] < min_minutes_for_labels:
            label = f"{label}, {int(row['minutes_played'])} mins"

        ax.annotate(
            label,
            (row["current_value_m"], row["predicted_value_m"]),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=8.2,
            color="#126b36",
            arrowprops={"arrowstyle": "-", "color": "#126b36", "lw": 0.6},
        )

    for _, row in negative_labels.iterrows():
        label = f"{row['player_name']}\n{format_eur_m(row['valuation_gap_m'])}"
        if row["minutes_played"] < min_minutes_for_labels:
            label = f"{label}, {int(row['minutes_played'])} mins"

        ax.annotate(
            label,
            (row["current_value_m"], row["predicted_value_m"]),
            xytext=(7, -14),
            textcoords="offset points",
            fontsize=8.2,
            color="#9b2525",
            arrowprops={"arrowstyle": "-", "color": "#9b2525", "lw": 0.6},
        )

    ax.text(
        axis_limit * 0.04,
        axis_limit * 0.92,
        "Above line = model rates higher",
        fontsize=10,
        color="#126b36",
        weight="bold",
    )
    ax.text(
        axis_limit * 0.52,
        axis_limit * 0.16,
        "Below line = model rates lower",
        fontsize=10,
        color="#9b2525",
        weight="bold",
    )

    most_undervalued = positive_labels.iloc[0] if len(positive_labels) else None
    most_overvalued = negative_labels.iloc[0] if len(negative_labels) else None
    summary_lines = []
    if most_undervalued is not None:
        summary_lines.append(
            f"Most undervalued by model: {format_summary_player(most_undervalued)}"
        )
    if most_overvalued is not None:
        summary_lines.append(
            f"Most overvalued by model: {format_summary_player(most_overvalued)}"
        )

    fig.text(
        0.11,
        0.025,
        "\n".join(summary_lines),
        ha="left",
        va="bottom",
        fontsize=10,
    )

    ax.set_title(f"{club_name}: Transfermarkt vs Model Value")
    ax.set_xlabel("Transfermarkt value (EUR m)")
    ax.set_ylabel("Model predicted value (EUR m)")
    ax.set_xlim(0, axis_limit)
    ax.set_ylim(0, axis_limit)
    ax.grid(True, alpha=0.22)
    fig.subplots_adjust(bottom=0.16)
    return fig, ax

## Table helpers


In [ ]:
def format_plain_eur_m(value):
    return f"EUR {value:.1f}m"


def format_gap_eur_m(value):
    sign = "+" if value > 0 else ""
    return f"{sign}EUR {value:.1f}m"


def make_club_valuation_table(data, club_name):
    club_data = (
        data.loc[data["season_club_name"].eq(club_name)]
        .sort_values("valuation_gap_m", ascending=False)
        .copy()
    )

    return pd.DataFrame({
        "Player": club_data["player_name"],
        "Pos": club_data["position"],
        "Mins": club_data["minutes_played"].astype(int),
        "TM value": club_data["current_value_m"].map(format_plain_eur_m),
        "Model value": club_data["predicted_value_m"].map(format_plain_eur_m),
        "Gap": club_data["valuation_gap_m"].map(format_gap_eur_m),
    })


def plot_club_valuation_table(data, club_name):
    club_data = (
        data.loc[data["season_club_name"].eq(club_name)]
        .sort_values("valuation_gap_m", ascending=False)
        .copy()
    )
    table_data = make_club_valuation_table(data, club_name)
    row_count = len(table_data)
    figure_height = max(7, row_count * 0.34 + 1.6)

    fig, ax = plt.subplots(figsize=(10, figure_height))
    ax.axis("off")
    ax.set_title(
        f"{club_name}: Transfermarkt vs Model Value",
        fontsize=16,
        weight="bold",
        pad=18,
    )

    table = ax.table(
        cellText=table_data.values,
        colLabels=table_data.columns,
        cellLoc="left",
        colLoc="left",
        loc="center",
        colWidths=[0.28, 0.12, 0.10, 0.16, 0.18, 0.16],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.25)

    for (row_index, column_index), cell in table.get_celld().items():
        cell.set_edgecolor("#dddddd")
        if row_index == 0:
            cell.set_facecolor("#222222")
            cell.set_text_props(color="white", weight="bold")
            continue

        gap_value = club_data.iloc[row_index - 1]["valuation_gap_m"]
        if column_index == 5:
            if gap_value > 0:
                cell.set_facecolor("#dff3e6")
                cell.set_text_props(color="#126b36", weight="bold")
            elif gap_value < 0:
                cell.set_facecolor("#f7dddd")
                cell.set_text_props(color="#9b2525", weight="bold")
        elif row_index % 2 == 0:
            cell.set_facecolor("#f7f7f7")

    return fig, ax


## Big Six plots

In [ ]:
saved_plot_files = []

for club_name in big_six_clubs:
    fig, ax = plot_club_valuation_scatter(big_six, club_name)
    output_path = figures_dir / f"{club_name.lower().replace(' ', '_')}_valuation_scatter.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    saved_plot_files.append(output_path)

saved_plot_files

## Big Six valuation tables

Each club also gets a table-only PNG. These are easier to use when writing scripts because every player is listed with Transfermarkt value, model value, and model gap.


In [ ]:
saved_table_files = []

for club_name in big_six_clubs:
    fig, ax = plot_club_valuation_table(big_six, club_name)
    output_path = figures_dir / f"{club_name.lower().replace(' ', '_')}_valuation_table.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    saved_table_files.append(output_path)

saved_table_files


## Combined Big Six plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 13))

for ax, club_name in zip(axes.flatten(), big_six_clubs):
    club_data = big_six.loc[big_six["season_club_name"].eq(club_name)].copy()
    max_value = max(
        club_data["current_value_m"].max(),
        club_data["predicted_value_m"].max(),
    )
    axis_limit = max_value * 1.12

    ax.fill_between(
        [0, axis_limit],
        [0, axis_limit],
        [axis_limit, axis_limit],
        color="#dff3e6",
        alpha=0.45,
    )
    ax.fill_between(
        [0, axis_limit],
        [0, 0],
        [0, axis_limit],
        color="#f7dddd",
        alpha=0.45,
    )
    ax.plot([0, axis_limit], [0, axis_limit], color="#222222", linestyle="--", linewidth=1)

    undervalued = club_data.loc[club_data["valuation_gap_m"].ge(0)]
    overvalued = club_data.loc[club_data["valuation_gap_m"].lt(0)]

    ax.scatter(
        undervalued["current_value_m"],
        undervalued["predicted_value_m"],
        s=undervalued["minutes_played"].clip(lower=90) / 18,
        c="#128a43",
        marker="^",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.6,
    )
    ax.scatter(
        overvalued["current_value_m"],
        overvalued["predicted_value_m"],
        s=overvalued["minutes_played"].clip(lower=90) / 18,
        c="#c73636",
        marker="v",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.6,
    )

    positive_labels, negative_labels = club_gap_extremes(
        club_data,
        club_name,
        top_each=1,
    )
    for _, row in positive_labels.iterrows():
        ax.annotate(
            row["player_name"],
            (row["current_value_m"], row["predicted_value_m"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            color="#126b36",
        )
    for _, row in negative_labels.iterrows():
        ax.annotate(
            row["player_name"],
            (row["current_value_m"], row["predicted_value_m"]),
            xytext=(5, -12),
            textcoords="offset points",
            fontsize=8,
            color="#9b2525",
        )

    most_under = format_summary_player(positive_labels.iloc[0]) if len(positive_labels) else "None"
    most_over = format_summary_player(negative_labels.iloc[0]) if len(negative_labels) else "None"
    ax.text(
        0.02,
        -0.22,
        f"Under: {most_under}\nOver: {most_over}",
        transform=ax.transAxes,
        fontsize=8,
        va="top",
    )

    ax.set_title(club_name)
    ax.set_xlim(0, axis_limit)
    ax.set_ylim(0, axis_limit)
    ax.grid(True, alpha=0.22)

fig.supxlabel("Transfermarkt value (EUR m)")
fig.supylabel("Model predicted value (EUR m)")
fig.suptitle("Big Six: Transfermarkt Value vs Model Value", fontsize=16)
fig.tight_layout(rect=[0, 0.05, 1, 0.96])

combined_output_path = figures_dir / "big_six_valuation_scatter_grid.png"
fig.savefig(combined_output_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

combined_output_path

## Notes

- `season_club_name` defines the club a player appeared for during the 2025/26 Premier League season.
- `current_club_name` remains the latest Transfermarkt snapshot and may differ for players who moved after appearing.
- These charts show model disagreement with Transfermarkt, not definitive proof that a player is undervalued or overvalued.